# AIRE325 TinyML Training — Fixed Auto-Movement No-Squeeze Model
Run all cells in Google Colab. Upload your CSV dataset file(s) when asked. This version creates `movementScore` automatically from `accX`, `accY`, `accZ` if needed.


In [ ]:
# AIRE325 TinyML Training — FIXED No-Squeeze Arduino Model
# This notebook trains a small CNN for Arduino Nano 33 BLE Sense.
# It avoids SQUEEZE and MAX_POOL ops because older Arduino TFLite Micro builds may not support them.

import os, re, json, zipfile
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)

WINDOW_SIZE = 10
FEATURE_COLUMNS = ["temperature", "ldr", "movementScore"]
CLASS_ORDER = ["Normal", "Caution", "Alert"]
OUTPUT_DIR = "tinyml_outputs_final_fixed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TensorFlow: 2.20.0


In [ ]:
# Upload either:
# 1) one combined CSV with a label/class/condition column, OR
# 2) separate CSV files named like normal.csv, caution.csv, alert.csv

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

for name, data_bytes in uploaded.items():
    with open(name, "wb") as f:
        f.write(data_bytes)

Saving final_dataset_demo_v2_profiled_CLEAN_UTF8.csv to final_dataset_demo_v2_profiled_CLEAN_UTF8.csv
Uploaded files: ['final_dataset_demo_v2_profiled_CLEAN_UTF8.csv']


In [ ]:
def normalize_col_name(c):
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

def find_column(df, candidates):
    norm_map = {normalize_col_name(c): c for c in df.columns}
    for cand in candidates:
        key = normalize_col_name(cand)
        if key in norm_map:
            return norm_map[key]
    for key, original in norm_map.items():
        for cand in candidates:
            if normalize_col_name(cand) in key:
                return original
    return None

def infer_label_from_filename(filename):
    lower = filename.lower()
    if "normal" in lower:
        return "Normal"
    if "caution" in lower:
        return "Caution"
    if "alert" in lower:
        return "Alert"
    return None

def compute_movement_score_from_acc(df, accx_col, accy_col, accz_col):
    """
    Creates movementScore from raw accelerometer readings.
    Formula matches Arduino logic approximately:
    sqrt((accX[t]-accX[t-1])^2 + (accY[t]-accY[t-1])^2 + (accZ[t]-accZ[t-1])^2)
    First row movement is 0.
    """
    acc = df[[accx_col, accy_col, accz_col]].apply(pd.to_numeric, errors="coerce")
    diff = acc.diff().fillna(0.0)
    movement = np.sqrt((diff ** 2).sum(axis=1))
    return movement.astype(np.float32)

def load_one_csv(filename):
    df = pd.read_csv(filename)
    df.columns = [str(c).strip() for c in df.columns]

    temp_col = find_column(df, ["temperature", "temp", "tempC", "temperatureC"])
    ldr_col = find_column(df, ["ldr", "light", "ldrRaw", "lightIntensity"])
    move_col = find_column(df, ["movementScore", "movement", "motion", "accelMovement"])
    label_col = find_column(df, ["label", "class", "condition", "target", "state"])

    # If movementScore is missing, create it from accX/accY/accZ.
    accx_col = find_column(df, ["accX", "accelerometerX", "x"])
    accy_col = find_column(df, ["accY", "accelerometerY", "y"])
    accz_col = find_column(df, ["accZ", "accelerometerZ", "z"])

    missing = []
    if temp_col is None: missing.append("temperature/tempC")
    if ldr_col is None: missing.append("ldr/ldrRaw")

    if move_col is None:
        if accx_col is None or accy_col is None or accz_col is None:
            missing.append("movementScore OR accX,accY,accZ")

    if missing:
        raise ValueError(f"{filename}: missing columns {missing}. Columns found: {list(df.columns)}")

    if move_col is not None:
        movement_values = pd.to_numeric(df[move_col], errors="coerce")
        print(f"{filename}: using existing movement column: {move_col}")
    else:
        movement_values = compute_movement_score_from_acc(df, accx_col, accy_col, accz_col)
        print(f"{filename}: created movementScore from {accx_col}, {accy_col}, {accz_col}")

    out = pd.DataFrame({
        "temperature": pd.to_numeric(df[temp_col], errors="coerce"),
        "ldr": pd.to_numeric(df[ldr_col], errors="coerce"),
        "movementScore": movement_values,
    })

    if label_col is not None:
        out["label"] = df[label_col].astype(str).str.strip()
    else:
        inferred = infer_label_from_filename(filename)
        if inferred is None:
            raise ValueError(f"{filename}: no label column and filename does not contain normal/caution/alert")
        out["label"] = inferred

    return out.dropna().reset_index(drop=True)

csv_files = [f for f in uploaded.keys() if f.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Upload CSV file(s), not only model/header files.")

data = pd.concat([load_one_csv(f) for f in csv_files], ignore_index=True)
label_map = {"normal":"Normal", "caution":"Caution", "alert":"Alert"}
data["label"] = data["label"].astype(str).str.strip().str.lower().map(label_map).fillna(data["label"])
data = data[data["label"].isin(CLASS_ORDER)].reset_index(drop=True)

print(data.head())
print("Shape:", data.shape)
print(data["label"].value_counts())



final_dataset_demo_v2_profiled_CLEAN_UTF8.csv: created movementScore from accX, accY, accZ
   temperature  ldr  movementScore   label
0        24.01  720       0.000000  Normal
1        23.74  674       0.003512  Normal
2        23.54  693       0.001647  Normal
3        23.78  692       0.007098  Normal
4        23.69  694       0.006057  Normal
Shape: (3000, 4)
label
Normal     1000
Caution    1000
Alert      1000
Name: count, dtype: int64


In [ ]:
# Scale features and encode labels
scaler = StandardScaler()
X_scaled = scaler.fit_transform(data[FEATURE_COLUMNS].values.astype(np.float32)).astype(np.float32)

label_to_id = {name: i for i, name in enumerate(CLASS_ORDER)}
y_raw = data["label"].map(label_to_id).values.astype(np.int64)

print("Feature means:", scaler.mean_)
print("Feature scales:", scaler.scale_)
print("Label map:", label_to_id)

Feature means: [3.14107000e+01 7.83054333e+02 1.68577867e-01]
Feature scales: [  6.3883517  146.76468711   0.22430462]
Label map: {'Normal': 0, 'Caution': 1, 'Alert': 2}


In [ ]:
# Create sliding windows. Each window predicts the label of the newest row in the window.
def make_windows(X, y, window_size=10):
    Xw, yw = [], []
    for i in range(len(X) - window_size + 1):
        Xw.append(X[i:i+window_size])
        yw.append(y[i+window_size-1])
    return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.int64)

X_win, y_win = make_windows(X_scaled, y_raw, WINDOW_SIZE)
X_win = X_win[..., np.newaxis].astype(np.float32)  # (samples, 10, 3, 1)

print("X_win shape:", X_win.shape)
print("y_win shape:", y_win.shape)
print("Input bytes per sample:", WINDOW_SIZE * len(FEATURE_COLUMNS) * 4)

X_train, X_test, y_train, y_test = train_test_split(
    X_win, y_win, test_size=0.2, random_state=42, stratify=y_win
)

X_win shape: (2991, 10, 3, 1)
y_win shape: (2991,)
Input bytes per sample: 120


In [ ]:
# Small Arduino-safe CNN.
# No Squeeze layer. No MaxPooling layer.
# Expected TFLite ops: CONV_2D, RESHAPE, FULLY_CONNECTED, SOFTMAX.

tf.keras.backend.clear_session()

model = tf.keras.Sequential([
    tf.keras.layers.Input(batch_shape=(1, WINDOW_SIZE, len(FEATURE_COLUMNS), 1)),
    tf.keras.layers.Conv2D(filters=4, kernel_size=(3, 1), activation="relu", padding="same"),
    tf.keras.layers.Reshape((WINDOW_SIZE * len(FEATURE_COLUMNS) * 4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(len(CLASS_ORDER), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=40,
    batch_size=1,
    verbose=1
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (1, 10, 3, 4)          │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (1, 120)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 8)                 │           968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (1, 3)                 │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,011 (3.95 KB)

 Trainable params: 1,011 (3.95 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9629 - loss: 0.1068 - val_accuracy: 0.9958 - val_loss: 0.0144
Epoch 2/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9974 - loss: 0.0089 - val_accuracy: 0.9979 - val_loss: 0.0125
Epoch 3/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9979 - loss: 0.0048 - val_accuracy: 0.9979 - val_loss: 0.0152
Epoch 4/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9995 - loss: 0.0029 - val_accuracy: 0.9979 - val_loss: 0.0119
Epoch 5/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9995 - loss: 0.0018 - val_accuracy: 0.9979 - val_loss: 0.0110
Epoch 6/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9995 - loss: 0.0014 - val_accuracy: 0.9979 - val_loss: 0.0103
Epoch 7/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 1.0000 - loss: 7.0570e-04 - val_accuracy: 0.9979 - val_loss: 0.0106
Epoch 8/40
1913/1913 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 1.0000 - loss: 4.249

In [ ]:
# Evaluate
loss, acc = model.evaluate(X_test, y_test, batch_size=1, verbose=0)
print("Test accuracy:", acc)

pred = model.predict(X_test, batch_size=1)
y_pred = np.argmax(pred, axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_ORDER))
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.9983305335044861
599/599 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00       199
     Caution       1.00      1.00      1.00       200
       Alert       1.00      0.99      1.00       200

    accuracy                           1.00       599
   macro avg       1.00      1.00      1.00       599
weighted avg       1.00      1.00      1.00       599

[[199   0   0]
 [  0 200   0]
 [  0   1 199]]


In [ ]:
# Convert to float TFLite model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = os.path.join(OUTPUT_DIR, "cabin_cnn_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

keras_path = os.path.join(OUTPUT_DIR, "cabin_cnn_model.keras")
model.save(keras_path)

print("TFLite model size:", len(tflite_model), "bytes")

try:
    print(tf.lite.experimental.Analyzer.analyze(model_content=tflite_model))
except Exception as e:
    print("Analyzer not available:", e)

Saved artifact at '/tmp/tmpxfwtogqq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 10, 3, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)
Captures:
  138370448549904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138370448551056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138370448549136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138370448548368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138370448550288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138370448549712: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model size: 6412 bytes
=== TFLite ModelAnalyzer ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the CONV_2D op takes
tensor #0 and tensor #6 and tensor #7 as input and produces tensor #8 as output.

Subgra

In [ ]:
# Generate Arduino header files

def bytes_to_c_array(data_bytes, var_name):
    lines = []
    lines.append("alignas(8) const unsigned char {}[] = {{".format(var_name))
    for i in range(0, len(data_bytes), 12):
        chunk = data_bytes[i:i+12]
        lines.append("  " + ", ".join("0x{:02x}".format(b) for b in chunk) + ",")
    lines.append("};")
    lines.append("const int {}_len = {};".format(var_name, len(data_bytes)))
    return "\n".join(lines)

model_h = "#ifndef MODEL_H\n#define MODEL_H\n\n" + bytes_to_c_array(tflite_model, "cabin_cnn_model") + "\n\n#endif\n"
with open(os.path.join(OUTPUT_DIR, "model.h"), "w") as f:
    f.write(model_h)

means = scaler.mean_.astype(float)
scales = scaler.scale_.astype(float)

scaler_h = """#ifndef SCALER_H
#define SCALER_H

#define WINDOW_SIZE {window_size}
#define NUM_FEATURES {num_features}

const float featureMeans[NUM_FEATURES] = {{{m0:.9f}f, {m1:.9f}f, {m2:.9f}f}};
const float featureScales[NUM_FEATURES] = {{{s0:.9f}f, {s1:.9f}f, {s2:.9f}f}};

#endif
""".format(
    window_size=WINDOW_SIZE,
    num_features=len(FEATURE_COLUMNS),
    m0=means[0], m1=means[1], m2=means[2],
    s0=scales[0], s1=scales[1], s2=scales[2]
)

with open(os.path.join(OUTPUT_DIR, "scaler.h"), "w") as f:
    f.write(scaler_h)

labels_h = """#ifndef LABELS_H
#define LABELS_H

const char* CLASS_NAMES[] = {\"Normal\", \"Caution\", \"Alert\"};

#endif
"""
with open(os.path.join(OUTPUT_DIR, "labels.h"), "w") as f:
    f.write(labels_h)

metadata = {
    "window_size": WINDOW_SIZE,
    "features": FEATURE_COLUMNS,
    "classes": CLASS_ORDER,
    "input_shape": [1, WINDOW_SIZE, len(FEATURE_COLUMNS), 1],
    "tflite_size_bytes": len(tflite_model),
    "feature_means": means.tolist(),
    "feature_scales": scales.tolist(),
    "expected_arduino_ops": ["Conv2D", "Reshape", "FullyConnected", "Softmax", "Relu fused"]
}
with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Generated files:")
for f in os.listdir(OUTPUT_DIR):
    print(" -", f)

Generated files:
 - cabin_cnn_model.keras
 - cabin_cnn_model.tflite
 - model.h
 - metadata.json
 - labels.h
 - scaler.h


In [ ]:
# Zip outputs for Arduino sketch folder
zip_path = "tinyml_outputs_final_fixed.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for fname in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, fname), arcname=fname)

print("Created", zip_path)
files.download(zip_path)

Created tinyml_outputs_final_fixed.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>